In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
import yaml

data_path = "../../experiment_data/separability_metrics/geometric_separability_random_mnist.csv"

random_data = pd.read_csv(data_path)

with open('../layer_orders_cross_layer.yml', 'r') as f:
    layer_order = yaml.safe_load(f)
        
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

In [2]:
full_data = random_data

In [3]:
resnet = full_data[(full_data["model"].str.contains("resnet")) & (full_data["split"] == "trainUval")].copy()

resnet["layer_idx"] = resnet["layer"].apply(lambda x: layer_order['resnet'].index(x))
resnet.sort_values("layer_idx", inplace=True)

resnet_accs = resnet[(resnet["layer"] == "a1")][["dataset", "model", "train_acc", "val_acc", "randomness"]]
resnet_accs.sort_values("randomness", inplace=True)

fig = make_subplots()
for i, row in resnet_accs.iterrows():
    color = color_seq[i % len(color_seq)]
    fig.add_trace(go.Scatter(x=[row["randomness"], row["randomness"]], y=[row["train_acc"], row["val_acc"]], mode="lines+markers", line = dict(color=color), name=f"{row['model']} ({row['randomness']})"))
    
fig.update_layout(title="ResNet Accuracies vs Random Label Proportion")
fig.update_xaxes(title_text="Randomness Proportion", nticks=23)
fig.update_yaxes(range=[0.0, 1.05], nticks=20)

In [4]:
fig = px.line(resnet, x="layer", y="neighbourhood_purity_20", color="randomness", symbol="model", color_discrete_sequence=px.colors.qualitative.Plotly, title="ResNet KNN Purity")
fig.update_xaxes(tickangle=80)
# fig.update_yaxes(type="log")

In [5]:
fig = px.line(resnet, x="layer", y="knn_accuracy_20", color="randomness", symbol="model", color_discrete_sequence=px.colors.qualitative.Plotly, title="KNN Classification Accuracy")
fig.update_xaxes(tickangle=80)
# fig.update_yaxes(type="log")